# In a nutshell

Experiment with prompts and different models on a subset of the annotated dataset.

# Setup

In [ ]:
import os
import sys
from pathlib import Path

os.chdir("..")
sys.path.append(".")

In [ ]:
Path.cwd()

In [3]:
from dotenv import load_dotenv

# Load API key from .env file if present.
load_dotenv()

# Not recommended: set API key manually
# %env VAR_NAME=value

True

# Settings

In [4]:
config_path = "llm/deepseek-r1-distill-llama-70b.config"
dataset_path = "data/eng_Latn_sample_for_llm_eval.parquet"

# Load the data

In [5]:
from datasets import Dataset

ds = Dataset.from_parquet(dataset_path)
print(f"Dataset size: {len(ds)} docs")
print(f"Columns:{ds.column_names}")
print(ds[:1])

Dataset size: 25 docs
Columns:['task_id', 'file_name', 'html', 'language', 'annotations', 'annotation_count', 'filename_warc', 'url', 'timestamp', 'collection']
{'task_id': [181409468], 'file_name': ['0ac1c6fa-57_ia_o_nz.html'], 'html': ['<!DOCTYPE html PUBLIC "-//W3C//DTD XHTML 1.0 Strict//EN" "http://www.w3.org/TR/xhtml1/DTD/xhtml1-strict.dtd">\n<html xmlns="http://www.w3.org/1999/xhtml"\nxml:lang="en" xmlns:fb="http://www.facebook.com/2008/fbml"\nxmlns:og="http://ogp.me/ns#" lang="en">\n<head>\n<meta http-equiv="Content-Type" content="text/html; charset=utf-8" />\n<title>ApplePlus iVehicle Accessories</title>\n<meta name="description" content="Get your car to be friend with your i-Devices. We\'ll build the bridge for you!" />\n<meta name="keywords" content="iPhone,iPad,iPod,accessories,iPad cases" />\n<meta name="robots" content="INDEX,FOLLOW" />\n<link rel="icon" href="http://www.appleplus.co.nz/media/favicon/default/favicon.ico" type="image/x-icon" />\n<link rel="shortcut icon" hr

# Call LLM annotator


In [17]:
import subprocess

args = [
    "--input",
    dataset_path,
    "--max_docs",
    "5",
    "--config",
    config_path,
]

process = subprocess.Popen(
    ["python", "-u", "-m", "llm.llm_annotator", *args],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    universal_newlines=True,
)

for line in process.stdout:
    print(line, end="")

# Wait for the process to complete
process.wait()

21:42:08 [INFO] Loaded 5 documents.
21:42:08 [DEBUG] Using proactor: IocpProactor
21:42:08 [INFO] Skipping task_id 181409468 (already processed)
21:42:08 [INFO] Skipping task_id 181410937 (already processed)
21:42:08 [INFO] Skipping task_id 181402673 (already processed)
21:42:08 [INFO] Skipping task_id 181406292 (already processed)
21:42:08 [INFO] Skipping task_id 181402115 (already processed)
21:42:08 [INFO] Processing complete.


0

# Load/merge anotations

In [ ]:
import json

config_name = Path(config_path).name.split(".")[0]

result_path = Path(config_path)
result_path = Path(f"llm/.annotations/{config_name}/results.jsonl")

with result_path.open(encoding="utf-8") as f:
    llm_annotations = [json.loads(line) for line in f]

print(f"Total docs annotated: {len(llm_annotations)}")

Total docs annotated: 15


In [8]:
import polars as pl

llm_annotations = pl.DataFrame(llm_annotations)
llm_annotations.head(1)

timestamp,config,model,task_id,duration_ms,annotations,total_tokens
str,str,str,i64,i64,list[struct[1]],i64
"""17/11/2025 19:03:54""","""deepseek-r1-distill-llama-70b""","""deepseek/deepseek-r1-distill-l…",181409468,203163,"[{""Get your car to be friend with your i-Devices. We'll build the bridge for you!""}, {""SGP Mobile Stand for Smart Phones""}, … {""$45.00""}]",18241


In [18]:
def merge_annotations(x: list[dict[str, str]]) -> str:
    return " ".join([k["text"] for k in x])


llm_annotations = llm_annotations.with_columns(
    pl.col("annotations")
    .map_elements(lambda x: merge_annotations(x), return_dtype=pl.Utf8)
    .alias("annotations_as_string")
)

llm_annotations.head(1)

timestamp,config,model,task_id,duration_ms,annotations,total_tokens,annotations_as_string
str,str,str,i64,i64,list[struct[1]],i64,str
"""20/11/2025 22:32:55""","""deepseek-r1-distill-llama-70b""","""deepseek/deepseek-r1-distill-l…",181402115,12154,"[{""Chrome Stand""}, {""This chrome shaving brush stand is the essential tool to allow your shaving brush to dry naturally.""}, … {""Kent Brushes""}]",48478,"""Chrome Stand This chrome shavi…"


# Compute metrics

In [10]:
from d2g_evaluation.evaluation.evaluate_human_vs_tool import HumanVsToolEvaluation

annotated_ids = set(llm_annotations["task_id"])
# sort both datasets by id, so that new llm annotation column is added correctly
ds_with_llm_annotations = ds.filter(lambda x: x["task_id"] in annotated_ids).sort(column_names=["task_id"])
llm_annotations = llm_annotations.sort(["task_id"])
llm_annotations_as_list = llm_annotations["annotations"].list.eval(pl.element().struct.field("text")).to_list()
ds_with_llm_annotations = ds_with_llm_annotations.add_column(
    "llm_annotations_as_string",
    llm_annotations["annotations_as_string"],
).add_column("llm_annotations", llm_annotations_as_list)


metrics = {
    "metric_name": "lcs_token_matching",
    "string_preprocessing_method": "normalize_string",
    "tokenization_method": "char_ngrams",
    "n": 3,
    "is_symmetric_forced": False,
}

eval_tool = HumanVsToolEvaluation()
eval_tool.INSUFFICIENT_ANNOTATIONS = 1

eval_result = eval_tool.evaluate(
    dataset=ds_with_llm_annotations,
    tool_column="llm_annotations_as_string",
    tool_name=llm_annotations[0]["model"],
    **metrics,
)

Evaluating with metric 'lcs_token_matching': 100%|##########| 15/15 [00:00<?, ? examples/s]

In [11]:
eval_result.column_names

['task_id',
 'file_name',
 'html',
 'language',
 'annotations',
 'annotation_count',
 'filename_warc',
 'url',
 'timestamp',
 'collection',
 'llm_annotations_as_string',
 'llm_annotations',
 'sample_result_human_vs_tool',
 'pairwise_results_human_vs_tool']

In [12]:
eval_result_df = eval_result.to_polars()
eval_stats = eval_result_df.select(
    [
        pl.col("task_id"),
        pl.col("sample_result_human_vs_tool")
        .struct.field("precision")
        .struct.field("mean")
        .round(3)
        .alias("precision"),
        pl.col("sample_result_human_vs_tool").struct.field("recall").struct.field("mean").round(3).alias("recall"),
        pl.col("sample_result_human_vs_tool").struct.field("f1").struct.field("mean").round(3).alias("f1"),
        pl.col("llm_annotations"),
        pl.col("annotations").list.eval(pl.element().struct[8].list[0].struct["text"]),
    ],
)

mean_p = round(eval_stats["precision"].mean(), 3)
mean_r = round(eval_stats["recall"].mean(), 3)
mean_f1 = round(eval_stats["f1"].mean(), 3)


# show all rows
pl.Config.set_tbl_rows(-1)
print(eval_stats[["task_id", "precision", "recall", "f1"]])
print(f"AVG:         {mean_p: < 10} |{mean_r: < 7} | {mean_f1: < 10}")

shape: (15, 4)
┌───────────┬───────────┬────────┬───────┐
│ task_id   ┆ precision ┆ recall ┆ f1    │
│ ---       ┆ ---       ┆ ---    ┆ ---   │
│ i64       ┆ f64       ┆ f64    ┆ f64   │
╞═══════════╪═══════════╪════════╪═══════╡
│ 181402115 ┆ 0.994     ┆ 0.162  ┆ 0.278 │
│ 181402673 ┆ 0.996     ┆ 0.822  ┆ 0.9   │
│ 181405931 ┆ 1.0       ┆ 0.847  ┆ 0.917 │
│ 181406292 ┆ 0.631     ┆ 0.999  ┆ 0.772 │
│ 181409340 ┆ 0.489     ┆ 0.843  ┆ 0.619 │
│ 181409392 ┆ 1.0       ┆ 1.0    ┆ 1.0   │
│ 181409394 ┆ 0.923     ┆ 0.813  ┆ 0.863 │
│ 181409430 ┆ 0.55      ┆ 0.712  ┆ 0.62  │
│ 181409468 ┆ 0.825     ┆ 0.559  ┆ 0.667 │
│ 181409541 ┆ 0.664     ┆ 0.976  ┆ 0.79  │
│ 181409543 ┆ 1.0       ┆ 1.0    ┆ 1.0   │
│ 181410937 ┆ 0.983     ┆ 0.947  ┆ 0.965 │
│ 181410959 ┆ 0.709     ┆ 0.753  ┆ 0.729 │
│ 181416738 ┆ 0.891     ┆ 0.837  ┆ 0.863 │
│ 181417072 ┆ 0.999     ┆ 0.847  ┆ 0.909 │
└───────────┴───────────┴────────┴───────┘
AVG:          0.844     | 0.808  |  0.793    


In [13]:
# Document length vs token cost
mean_tokens_consumed = round(llm_annotations["total_tokens"].mean())
mean_doclength_chars = round(ds_with_llm_annotations.to_polars()["html"].map_elements(lambda x: len(x)).mean())
compression_ratio = round(mean_doclength_chars / mean_tokens_consumed, 2)

print(
    f"Avg doc length: {mean_doclength_chars:,} chars | \
Avg tokens consumed: {mean_tokens_consumed:,} | \
Compression ratio: {compression_ratio}"
)

Avg doc length: 81,207 chars | Avg tokens consumed: 24,745 | Compression ratio: 3.28


# Explore low-performers

In [ ]:
low_performers = eval_stats.filter(pl.col("f1") < 0.7)  # noqa

with pl.Config(fmt_str_lengths=10**5, fmt_table_cell_list_len=10**5, tbl_cols=-1, tbl_rows=-1):
    print(low_performers["task_id", "precision", "recall", "f1", "llm_annotations"])

shape: (4, 5)
┌───────────┬───────────┬────────┬───────┬─────────────────────────────────────────────────────────┐
│ task_id   ┆ precision ┆ recall ┆ f1    ┆ llm_annotations                                         │
│ ---       ┆ ---       ┆ ---    ┆ ---   ┆ ---                                                     │
│ i64       ┆ f64       ┆ f64    ┆ f64   ┆ list[str]                                               │
╞═══════════╪═══════════╪════════╪═══════╪═════════════════════════════════════════════════════════╡
│ 181402115 ┆ 0.994     ┆ 0.162  ┆ 0.278 ┆ ["Chrome Stand", "This chrome shaving brush stand is    │
│           ┆           ┆        ┆       ┆ the essential tool to allow your shaving brush to dry   │
│           ┆           ┆        ┆       ┆ naturally.", "Award-Winning -  G B Kent & Sons Ltd,     │
│           ┆           ┆        ┆       ┆ have been manufacturers of brushes since the eighteenth │
│           ┆           ┆        ┆       ┆ century, and are one of the oldest